In [95]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error
import joblib

In [96]:
confirmed = pd.read_csv("/time_series_covid19_confirmed_global.csv")
deaths = pd.read_csv("/time_series_covid19_deaths_global.csv")
recovered = pd.read_csv("/time_series_covid19_recovered_global.csv")

In [97]:
def melt_df(df):
    return df.melt(
        id_vars=["Province/State", "Country/Region", "Lat", "Long"],
        var_name="date",
        value_name="value"
    )

confirmed = melt_df(confirmed).rename(columns={"value": "confirmed"})
deaths = melt_df(deaths).rename(columns={"value": "deaths"})
recovered = melt_df(recovered).rename(columns={"value": "recovered"})

In [98]:
df = confirmed.merge(deaths, on=["Province/State","Country/Region","Lat","Long","date"])
df = df.merge(recovered, on=["Province/State","Country/Region","Lat","Long","date"])

In [99]:
df['date'] = pd.to_datetime(df['date'])

/tmp/ipykernel_32137/3532345252.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])


In [100]:
df = df.groupby(['Country/Region','date']).sum().reset_index()

In [101]:
df['cases'] = df.groupby('Country/Region')['confirmed'].diff().fillna(0)
df['deaths_daily'] = df.groupby('Country/Region')['deaths'].diff().fillna(0)
df['recovered_daily'] = df.groupby('Country/Region')['recovered'].diff().fillna(0)

df[['cases','deaths_daily','recovered_daily']] = df[['cases','deaths_daily','recovered_daily']].clip(lower=0)
df['active'] = df['confirmed'] - df['deaths'] - df['recovered']

df['next_day_cases'] = df.groupby('Country/Region')['cases'].shift(-1)

In [102]:
df['mobility'] = np.random.uniform(0.3, 1.0, size=len(df))

# Case growth rate
df['case_growth'] = df.groupby('Country/Region')['cases'].pct_change().fillna(0)

# Risk = log-scaled cases * mobility * growth factor
df['risk'] = np.log1p(df['cases']) * df['mobility'] * (1 + df['case_growth'])
df['risk'] = df['risk'].clip(0,1)  # keeps values between 0 and 1

In [103]:
def hotspot_dynamic(row):
    if row['risk'] > 0.2:
        return 2  # high hotspot
    elif row['risk'] > 0.05:
        return 1  # medium hotspot
    else:
        return 0  # low

df['hotspot'] = df.apply(hotspot_dynamic, axis=1)

In [104]:
df = df.dropna()

In [105]:
# Train Regression Model
reg_model = MultiOutputRegressor(
    HistGradientBoostingRegressor(max_iter=200, random_state=42)
)
reg_model.fit(X_train_new, y_train_r_new)

# Train Classification Model
cls_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)
cls_model.fit(X_train_new, y_train_c_new)

print('Models trained successfully using X_train_new.')

Models trained successfully using X_train_new.


In [106]:
for lag in range(1, 4): # Create lags for 1, 2, and 3 days prior
    df[f'cases_lag_{lag}'] = df.groupby('Country/Region')['cases'].shift(lag).fillna(0)
    df[f'deaths_daily_lag_{lag}'] = df.groupby('Country/Region')['deaths_daily'].shift(lag).fillna(0)
    df[f'recovered_daily_lag_{lag}'] = df.groupby('Country/Region')['recovered_daily'].shift(lag).fillna(0)
    df[f'active_lag_{lag}'] = df.groupby('Country/Region')['active'].shift(lag).fillna(0)

# Drop rows that might have NaNs introduced by shifting at the beginning of each group
df = df.dropna().reset_index(drop=True)

display(df.head())

,Country/Region,date,Province/State,Lat,Long,confirmed,deaths,recovered,cases,deaths_daily,...,recovered_daily_lag_1,active_lag_1,cases_lag_2,deaths_daily_lag_2,recovered_daily_lag_2,active_lag_2,cases_lag_3,deaths_daily_lag_3,recovered_daily_lag_3,active_lag_3
0,Afghanistan,2020-01-22,0,33.93911,67.709953,0,0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Afghanistan,2020-01-23,0,33.93911,67.709953,0,0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Afghanistan,2020-01-24,0,33.93911,67.709953,0,0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Afghanistan,2020-01-25,0,33.93911,67.709953,0,0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Afghanistan,2020-01-26,0,33.93911,67.709953,0,0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [107]:
# 1. Create time-based features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek
df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)

# 2. Create lagged features
for lag in range(1, 4):
    df[f'cases_lag_{lag}'] = df.groupby('Country/Region')['cases'].shift(lag)
    df[f'deaths_daily_lag_{lag}'] = df.groupby('Country/Region')['deaths_daily'].shift(lag)
    df[f'recovered_daily_lag_{lag}'] = df.groupby('Country/Region')['recovered_daily'].shift(lag)
    df[f'active_lag_{lag}'] = df.groupby('Country/Region')['active'].shift(lag)

# 3. Drop rows with NaNs from shifting and select features
df_clean = df.dropna().reset_index(drop=True)

features = [
    'cases', 'deaths_daily', 'recovered_daily', 'active', 'mobility',
    'cases_lag_1', 'cases_lag_2', 'cases_lag_3',
    'deaths_daily_lag_1', 'deaths_daily_lag_2', 'deaths_daily_lag_3',
    'recovered_daily_lag_1', 'recovered_daily_lag_2', 'recovered_daily_lag_3',
    'active_lag_1', 'active_lag_2', 'active_lag_3',
    'year', 'month', 'day', 'dayofweek', 'weekofyear'
]

X_new = df_clean[features]
y_reg_new = df_clean[['next_day_cases', 'risk']]
y_cls_new = df_clean['hotspot']

# 4. Split data
from sklearn.model_selection import train_test_split
X_train_new, X_test_new, y_train_r_new, y_test_r_new, y_train_c_new, y_test_c_new = train_test_split(
    X_new, y_reg_new, y_cls_new, test_size=0.2, random_state=42
)

print(f'Data prepared. Training features: {X_train_new.shape[1]}')

Data prepared. Training features: 22


In [108]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred_r_new = reg_model.predict(X_test_new)
pred_cases_new = y_pred_r_new[:,0]
pred_risk_new = y_pred_r_new[:,1]
pred_hotspot_new = cls_model.predict(X_test_new)

mse_cases_new = mean_squared_error(y_test_r_new['next_day_cases'], pred_cases_new)
mse_risk_new = mean_squared_error(y_test_r_new['risk'], pred_risk_new)

r2_cases_new = r2_score(y_test_r_new['next_day_cases'], pred_cases_new)
r2_risk_new = r2_score(y_test_r_new['risk'], pred_risk_new)

rmse_cases_new = np.sqrt(mse_cases_new)

print(f"Cases MSE (New Features): {mse_cases_new:.4f}")
print(f"Cases RMSE (New Features): {rmse_cases_new:.4f}")
print(f"Cases R2 Score (New Features): {r2_cases_new * 100:.2f}%")
print('-' * 30)
print(f"Risk MSE (New Features): {mse_risk_new:.8f}")
print(f"Risk R2 Score (New Features): {r2_risk_new * 100:.2f}%")

Cases MSE (New Features): 55093314.1510
Cases RMSE (New Features): 7422.4871
Cases R2 Score (New Features): 83.59%
------------------------------
Risk MSE (New Features): 0.00021546
Risk R2 Score (New Features): 99.91%


In [109]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek
df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)

display(df.head())

,Country/Region,date,Province/State,Lat,Long,confirmed,deaths,recovered,cases,deaths_daily,...,active_lag_2,cases_lag_3,deaths_daily_lag_3,recovered_daily_lag_3,active_lag_3,year,month,day,dayofweek,weekofyear
0,Afghanistan,2020-01-22,0,33.93911,67.709953,0,0,0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,2020,1,22,2,4
1,Afghanistan,2020-01-23,0,33.93911,67.709953,0,0,0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,2020,1,23,3,4
2,Afghanistan,2020-01-24,0,33.93911,67.709953,0,0,0,0.0,0.0,...,0.0,NaN,NaN,NaN,NaN,2020,1,24,4,4
3,Afghanistan,2020-01-25,0,33.93911,67.709953,0,0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2020,1,25,5,4
4,Afghanistan,2020-01-26,0,33.93911,67.709953,0,0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2020,1,26,6,4


In [110]:
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
import joblib

# 1. Advanced TimeSeries Cross-Validation & Hyperparameter Tuning
tscv = TimeSeriesSplit(n_splits=3)

print("--- Optimizing Regression Model ---")
reg_param_grid = {
    'estimator__max_iter': [100, 200, 300],
    'estimator__learning_rate': [0.01, 0.1, 0.2],
    'estimator__max_depth': [3, 5, 10]
}
reg_search = RandomizedSearchCV(
    MultiOutputRegressor(HistGradientBoostingRegressor(random_state=42)),
    param_distributions=reg_param_grid,
    n_iter=5,
    cv=tscv,
    scoring='r2',
    n_jobs=-1
)
reg_search.fit(X_train_new, y_train_r_new)
best_reg_model = reg_search.best_estimator_

print("--- Optimizing Classification Model ---")
cls_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}
cls_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_distributions=cls_param_grid,
    n_iter=5,
    cv=tscv,
    scoring='f1_macro',
    n_jobs=-1
)
cls_search.fit(X_train_new, y_train_c_new)
best_cls_model = cls_search.best_estimator_

print(f"Best Reg R2: {reg_search.best_score_:.4f}")
print(f"Best Cls F1: {cls_search.best_score_:.4f}")

--- Optimizing Regression Model ---
--- Optimizing Classification Model ---
Best Reg R2: 0.8863
Best Cls F1: 0.8543


In [114]:
import os

# 1. Verify existence of production artifacts
artifacts = ['covid_regressor_v1.joblib', 'covid_classifier_v1.joblib', 'model_features_list.joblib']
for art in artifacts:
    status = 'EXISTS' if os.path.exists(art) else 'MISSING'
    print(f"Artifact {art}: {status}")

# 2. Final Test of the Production Inference Function
try:
    test_sample = X_test_new.head(1)
    results = prediction_result(test_sample)
    print("\n--- Final Production Check ---")
    print(f"Predicted Cases: {results['predicted_cases'][0]:.2f}")
    print(f"Predicted Risk Score: {results['predicted_risk'][0]:.4f}")
    print(f"Hotspot Level: {results['hotspot_level'][0]}")
    print("Status: PIPELINE FULLY FUNCTIONAL")
except Exception as e:
    print(f"Validation failed: {e}")

Artifact covid_regressor_v1.joblib: EXISTS
Artifact covid_classifier_v1.joblib: EXISTS
Artifact model_features_list.joblib: EXISTS

--- Final Production Check ---
Predicted Cases: 731.82
Predicted Risk Score: 0.9977
Hotspot Level: 2
Status: PIPELINE FULLY FUNCTIONAL


In [111]:
# 2. Exporting Models for Production Use
joblib.dump(best_reg_model, 'covid_regressor_v1.joblib')
joblib.dump(best_cls_model, 'covid_classifier_v1.joblib')
joblib.dump(features, 'model_features_list.joblib')

print("Models and feature list saved as .joblib files for industry deployment.")

Models and feature list saved as .joblib files for industry deployment.


In [112]:
def prediction_result (input_data, reg_path='covid_regressor_v1.joblib', cls_path='covid_classifier_v1.joblib'):
    """
    Industry-level inference function that loads saved models and predicts.
    """
    try:
        reg = joblib.load(reg_path)
        cls = joblib.load(cls_path)

        # Regression Prediction
        preds_r = reg.predict(input_data)
        # Classification Prediction
        preds_c = cls.predict(input_data)

        return {
            "predicted_cases": preds_r[:, 0],
            "predicted_risk": preds_r[:, 1],
            "hotspot_level": preds_c
        }
    except Exception as e:
        return {"error": str(e)}

# Quick test with production-style call
latest_sample = X_test_new.head(1)
print("prediction_result:", prediction_result(latest_sample))

prediction_result: {'predicted_cases': array([731.82019264]), 'predicted_risk': array([0.99767404]), 'hotspot_level': array([2])}


In [ ]:
import os

model_dir = os.path.join('backend', 'app', 'model')
os.makedirs(model_dir, exist_ok=True)

joblib.dump(best_reg_model, os.path.join(model_dir, 'regressor.joblib'))
joblib.dump(best_cls_model, os.path.join(model_dir, 'classifier.joblib'))

print('Models saved successfully in backend/app/model as .joblib files.')

Models saved successfully as .pkl files.
